# B04a: STFT and Spectrograms (Quick Version)

**Duration**: ~25 minutes  
**Prerequisites**: A02, A03

> 💡 **Quick Version**: Uses `scipy.signal.stft` and `scipy.signal.spectrogram` directly. See the full tutorial for step-by-step implementation.


## Table of Contents

1. [Introduction — Why STFT?](#section-1-introduction)
2. [STFT Computation](#section-2-stft)
3. [Parameter Exploration](#section-3-parameters)
4. [Exercises](#section-4-exercises)
5. [Summary](#summary)
6. [External Resources](#external-resources)
7. [Discussion Questions](#discussion-questions)


In [ ]:
# =============================================================================
# Imports
# =============================================================================

import sys
from pathlib import Path

import numpy as np
from numpy.typing import NDArray
import matplotlib.pyplot as plt
from scipy.signal import stft, spectrogram, get_window
from scipy.fft import fft, fftfreq

src_path = Path.cwd().parent.parent.parent / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from colors import COLORS

plt.rcParams["figure.dpi"] = 100
plt.rcParams["font.size"] = 11

np.random.seed(42)
fs = 256  # Standard EEG sampling rate
print("Imports loaded ✓")


---

<a id="section-1-introduction"></a>
## 1. Introduction — Why STFT?

The **Fourier transform** assumes stationarity: frequency content is constant over time.
EEG signals violate this: alpha bursts appear and disappear, cognitive events modulate oscillations.

**Solution**: The Short-Time Fourier Transform slides a window over the signal.
$$\text{STFT}(t, f) = \int x(\tau) \cdot w(\tau - t) \cdot e^{-i 2\pi f \tau} \, d\tau$$

**Key trade-off** (Heisenberg uncertainty):
- Short window → good time resolution, poor frequency resolution  
- Long window → good frequency resolution, poor time resolution


In [ ]:
# Non-stationary signal: frequency changes over time
t = np.arange(0, 4.0, 1/fs)

# Chirp: frequency sweeps from 5 Hz to 30 Hz
freq_sweep = 5 + 25 * t / 4
chirp = np.sin(2 * np.pi * np.cumsum(freq_sweep) / fs)

# FFT (loses temporal info)
freqs_fft = fftfreq(len(t), 1/fs)
magnitude = np.abs(fft(chirp))[:len(t)//2]

# STFT (preserves temporal info)
f_stft, t_stft, Zxx = stft(chirp, fs, nperseg=128, noverlap=64, window='hann')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(t, chirp, color=COLORS["signal_1"], lw=0.8)
axes[0].set_title("Signal (Chirp: 5→30 Hz)", fontweight="bold")
axes[0].set_xlabel("Time (s)")

axes[1].plot(freqs_fft[:len(t)//2], magnitude, color=COLORS["signal_2"], lw=1.5)
axes[1].set_xlim(0, 50)
axes[1].set_title("FFT — Loses Temporal Info", fontweight="bold")
axes[1].set_xlabel("Frequency (Hz)")

axes[2].pcolormesh(t_stft, f_stft, np.abs(Zxx), cmap='hot', shading='gouraud')
axes[2].set_ylim(0, 50)
axes[2].set_title("STFT Spectrogram — Time-Frequency", fontweight="bold")
axes[2].set_xlabel("Time (s)")
axes[2].set_ylabel("Frequency (Hz)")
plt.tight_layout()
plt.show()
print("→ The STFT reveals the frequency sweep that the FFT hides")


---

<a id="section-2-stft"></a>
## 2. STFT on EEG-like Signals

Applying the STFT to a realistic multi-component EEG signal.


In [ ]:
np.random.seed(42)
t_eeg = np.arange(0, 8.0, 1/fs)

# Simulate state changes
alpha_on  = np.zeros(len(t_eeg))
alpha_on[(t_eeg > 1) & (t_eeg < 3)] = 1.0
alpha_on[(t_eeg > 5) & (t_eeg < 7)] = 0.8

beta_on = np.zeros(len(t_eeg))
beta_on[(t_eeg > 3.5) & (t_eeg < 5)] = 1.0

eeg = (alpha_on * np.sin(2 * np.pi * 10 * t_eeg) +
       beta_on  * np.sin(2 * np.pi * 20 * t_eeg) +
       0.5 * np.sin(2 * np.pi * 3 * t_eeg) +
       0.3 * np.random.randn(len(t_eeg)))

# STFT spectrogram
f_spec, t_spec, Sxx = spectrogram(eeg, fs, nperseg=256, noverlap=200, window='hann')

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(t_eeg, eeg, color=COLORS["signal_1"], lw=0.7, alpha=0.8)
axes[0].set_ylabel("Amplitude (a.u.)")
axes[0].set_title("Simulated EEG (Alpha + Beta state changes)", fontweight="bold")
axes[0].grid(True, alpha=0.3)

pcm = axes[1].pcolormesh(t_spec, f_spec, 10 * np.log10(Sxx + 1e-10),
                          cmap='inferno', shading='gouraud', vmin=-40, vmax=10)
axes[1].set_ylim(0, 40)
axes[1].set_ylabel("Frequency (Hz)")
axes[1].set_xlabel("Time (s)")
axes[1].set_title("Spectrogram (Power in dB)", fontweight="bold")
plt.colorbar(pcm, ax=axes[1], label="Power (dB)")
# Mark EEG bands
for freq, label in [(4, "δ/θ"), (8, "α"), (13, "β"), (30, "γ")]:
    axes[1].axhline(freq, color="white", lw=0.5, alpha=0.5)
plt.tight_layout()
plt.show()


---

<a id="section-3-parameters"></a>
## 3. Parameter Exploration

Window length determines the time-frequency resolution trade-off.


In [ ]:
window_lengths = [32, 128, 512]  # samples at 256 Hz → 125 ms, 500 ms, 2 s

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, nperseg in zip(axes, window_lengths):
    f_p, t_p, Sxx_p = spectrogram(eeg, fs, nperseg=nperseg, noverlap=nperseg//2, window='hann')
    duration_ms = nperseg / fs * 1000
    freq_res = fs / nperseg
    ax.pcolormesh(t_p, f_p, 10 * np.log10(Sxx_p + 1e-10), cmap='inferno', shading='gouraud', vmin=-40, vmax=10)
    ax.set_ylim(0, 40)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Frequency (Hz)")
    ax.set_title(f"Window: {nperseg} samples\n({duration_ms:.0f} ms, Δf = {freq_res:.1f} Hz)", fontweight="bold")

plt.suptitle("Time-Frequency Resolution Trade-off", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()
print("Short window → time resolution ↑, frequency resolution ↓")
print("Long window  → frequency resolution ↑, time resolution ↓")


---

<a id="section-4-exercises"></a>
## 4. Exercises

### 🎯 Exercise 1: Window Types

**Difficulty**: 🟢 Easy  
**Task**: Compare spectrograms using 'hann', 'hamming', and 'boxcar' (rectangular) windows of the same length. What do you notice about spectral leakage?

```python
windows = ['hann', 'hamming', 'boxcar']
# For each window: spectrogram(eeg, fs, nperseg=256, window=w)
# Compare side-by-side
```

### 🎯 Exercise 2: Alpha Power Time-Course

**Difficulty**: 🟡 Medium  
**Task**: From the spectrogram, extract the average power in the alpha band (8–13 Hz) for each time frame to create an alpha power time-course. Compare with the envelope from B03a.

```python
# Hint: mask f_spec to 8-13 Hz range, then average Sxx over that range
alpha_mask = (f_spec >= 8) & (f_spec <= 13)
alpha_power_stft = Sxx[alpha_mask, :].mean(axis=0)
```


---

<a id="summary"></a>
## 5. Summary

### Key Concepts

| Parameter | Effect |
|-----------|--------|
| Window length (`nperseg`) | Longer → better frequency res, worse time res |
| Overlap (`noverlap`) | More overlap → smoother time axis, more computation |
| Window type | Hann recommended — reduces spectral leakage |

### Time-Frequency Resolution

$$\Delta t = \frac{N_\text{window}}{f_s} \qquad \Delta f = \frac{f_s}{N_\text{window}}$$

These multiply to a constant: $\Delta t \cdot \Delta f = 1$ (Heisenberg uncertainty).


---

<a id="external-resources"></a>
## 6. External Resources

### 📚 Scientific References

- **Cohen, M.X. (2014)** - *Analyzing Neural Time Series Data* — Chapter 13: Time-frequency analysis
- **Allen, J.B. & Rabiner, L.R. (1977)** - *A unified approach to short-time Fourier analysis*

### 🎧 NotebookLM Resources

*(NotebookLM links will be added once this notebook is finalized)*


---

<a id="discussion-questions"></a>
## 7. Discussion Questions

1. **Resolution trade-off**: You need to detect 50 ms alpha bursts AND resolve individual frequency bands (alpha vs beta). Is this achievable with a single STFT? What approach might help?

2. **Overlap intuition**: With 50% overlap, what happens to the effective temporal resolution? With 90% overlap, how many more frames do you get?

3. **STFT vs Wavelets**: For studying event-related gamma (30–80 Hz) responses lasting 50 ms, would you prefer STFT or Morlet wavelets (B04b)? Why?
